# Shri Ramswaroop Memorial University (SRMU)
## Department of Computer Science & Engineering
### B.Tech (CSE) – VII Semester | Deep Learning Lab

---

# **EXPERIMENT 5**
## **Title**: Implementation of Logistic Regression Using TensorFlow.

---

### **1. Aim / Objective**
1. To understand the mathematical principles and computational architecture of **Logistic Regression** as the foundational single-neuron building block of deep neural networks.
2. To implement Logistic Regression in TensorFlow using two distinct paradigms:
   - **Low-Level TensorFlow (From First Principles)**:
     - Initialize trainable parameter tensors (`tf.Variable`) for weights $\mathbf{w}$ and bias $b$.
     - Implement forward hypothesis computation using `tf.sigmoid`.
     - Implement custom **Binary Cross-Entropy Loss** with numerical clipping.
     - Leverage `tf.GradientTape` for automatic gradient computation ($\nabla_{\mathbf{w}}\mathcal{L}, \nabla_b\mathcal{L}$).
     - Execute a manual training loop using gradient descent parameter updates.
   - **High-Level TensorFlow / Keras**:
     - Construct equivalent architecture using `tf.keras.Sequential` and `tf.keras.layers.Dense(1, activation='sigmoid')`.
     - Compile using `tf.keras.optimizers.Adam` and `binary_crossentropy`.
     - Train using `model.fit()` with validation split tracking.
3. To evaluate both models on a standardized binary classification dataset:
   - Confusion Matrix (TP, FP, TN, FN)
   - Classification Report (Accuracy, Precision, Recall, F1-Score)
   - Receiver Operating Characteristic (ROC) curve and AUC
   - 2D **Decision Boundary Visualization** illustrating the separating hyperplane.
4. To compare Low-Level TensorFlow versus High-Level Keras implementations in terms of control, customization, and developer productivity.

---

### **2. Prerequisites & Software Requirements**
* **Language**: Python 3.8+
* **Libraries**:
  - `tensorflow` (>= 2.x): Tensor computations, automatic differentiation, Keras API.
  - `numpy`: Array manipulation.
  - `matplotlib` & `seaborn`: Loss curves, confusion matrix heatmaps, decision boundaries.
  - `scikit-learn`: Synthetic dataset generation (`make_classification`), feature standardization, and evaluation metrics.

---

### **3. Mathematical Foundations**

#### **3.1 The Logistic Regression Model**
Logistic Regression is a supervised learning algorithm for binary classification ($y \in \{0, 1\}$).
For an input vector $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$:

1. **Linear Combination (Logits)**:
   $$z = \mathbf{w}^T \mathbf{x} + b = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

2. **Sigmoid Hypothesis (Probability Mapping)**:
   $$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}} = P(y=1 \mid \mathbf{x})$$
   - If $\hat{y} \ge 0.5 \implies \text{Class } 1$
   - If $\hat{y} < 0.5 \implies \text{Class } 0$

3. **Decision Boundary Equation**:
   The decision threshold occurs where $\hat{y} = 0.5 \iff z = 0$:
   $$\mathbf{w}^T \mathbf{x} + b = 0 \implies w_1 x_1 + w_2 x_2 + b = 0 \implies x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$
   This defines a linear hyperplane separating the two classes in feature space.

---

#### **3.2 Loss Function: Binary Cross-Entropy vs Mean Squared Error**

```
+-------------------------------------------------------------------------------+
|  WHY NOT MEAN SQUARED ERROR (MSE)?                                            |
|  J_MSE = (1/m) * sum( (y - sigmoid(z))^2 )                                    |
|  Due to the non-linear sigmoid function, J_MSE is NON-CONVEX.                 |
|  It contains multiple local minima and saddle points where gradients vanish!  |
+-------------------------------------------------------------------------------+
|  BINARY CROSS-ENTROPY (LOG-LOSS):                                             |
|  L(w, b) = -(1/m) * sum [ y * ln(y_hat) + (1 - y) * ln(1 - y_hat) ]           |
|  Strictly CONVEX with respect to weights w and bias b.                        |
|  Guarantees that Gradient Descent will converge to the GLOBAL MINIMUM!        |
+-------------------------------------------------------------------------------+
```

#### **3.3 Analytical Gradients (Maximum Likelihood Estimation)**
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$
$$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})$$

**Gradient Descent Update Rule**:
$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{w}}$$
$$b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$


---
### **4. Step-by-Step Implementation**

#### **Step 1: Environment Verification & Library Imports**


In [ ]:
# Step 1: Import Modules
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

print(f"TensorFlow Version: {tf.__version__}")
print("[SUCCESS] Core modules imported successfully!")


#### **Step 2: Dataset Generation & Preprocessing**
We generate a binary classification dataset with 2 informative features to facilitate 2D visualization of the decision boundary.


In [ ]:
# Generate Synthetic 2D Binary Classification Dataset
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    flip_y=0.05,       # 5% label noise
    class_sep=1.5,     # Well separated
    random_state=42
)

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)

# Feature Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to TensorFlow float32 Tensors
X_train_tf = tf.constant(X_train_scaled, dtype=tf.float32)
y_train_tf = tf.constant(y_train.reshape(-1, 1), dtype=tf.float32)
X_test_tf  = tf.constant(X_test_scaled, dtype=tf.float32)
y_test_tf  = tf.constant(y_test.reshape(-1, 1), dtype=tf.float32)

print(f"Dataset summary: Total = {len(X_raw)}, Train = {len(X_train)}, Test = {len(X_test)}")
print(f"X_train Tensor Shape: {X_train_tf.shape}, y_train Tensor Shape: {y_train_tf.shape}")


#### **Step 3: Visualizing the Dataset Distribution**


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    X_train_scaled[y_train == 0, 0], X_train_scaled[y_train == 0, 1],
    color='#e05d44', label='Class 0 (Negative)', alpha=0.7, edgecolors='k', s=45
)
plt.scatter(
    X_train_scaled[y_train == 1, 0], X_train_scaled[y_train == 1, 1],
    color='#2b5c8f', label='Class 1 (Positive)', alpha=0.7, edgecolors='k', s=45
)
plt.title("Standardized Training Dataset (2 Features)", fontsize=13, fontweight='bold')
plt.xlabel("Feature 1 (Standardized)")
plt.ylabel("Feature 2 (Standardized)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


#### **Step 4: Low-Level TensorFlow Implementation (Custom Model Class)**
We construct a custom Logistic Regression model from scratch using TensorFlow primitives:
- Trainable weights $\mathbf{W} \in \mathbb{R}^{2 	imes 1}$ and bias $b \in \mathbb{R}^{1}$ stored as `tf.Variable`.
- Forward pass: $\hat{y} = \sigma(\mathbf{X}\mathbf{W} + b)$.
- Loss: Binary Cross-Entropy with `tf.clip_by_value` to prevent $\ln(0)$ numerical instability.
- Backward Pass & Optimization: `tf.GradientTape()` to calculate $rac{\partial \mathcal{L}}{\partial \mathbf{W}}$ and $rac{\partial \mathcal{L}}{\partial b}$, updated via `assign_sub`.


In [ ]:
class LowLevelLogisticRegression:
    """
    Logistic Regression built from first principles with Low-Level TensorFlow.
    """
    def __init__(self, n_features, learning_rate=0.1, seed=42):
        tf.random.set_seed(seed)
        self.lr = learning_rate
        
        # Trainable Parameters
        # Weights initialized from Gaussian normal distribution
        self.W = tf.Variable(tf.random.normal(shape=(n_features, 1), stddev=0.1, dtype=tf.float32), name="weights")
        # Bias initialized to zero
        self.b = tf.Variable(tf.zeros(shape=(1,), dtype=tf.float32), name="bias")
        
        self.loss_history = []

    def forward(self, X):
        """Forward Pass: y_hat = sigmoid(X @ W + b)"""
        z = tf.matmul(X, self.W) + self.b
        return tf.sigmoid(z)

    def compute_loss(self, y_true, y_pred):
        """Binary Cross-Entropy Loss with numerical stability clipping."""
        eps = 1e-15
        y_pred_clipped = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        loss = - tf.reduce_mean(
            y_true * tf.math.log(y_pred_clipped) + (1.0 - y_true) * tf.math.log(1.0 - y_pred_clipped)
        )
        return loss

    def train_step(self, X, y):
        """Single Gradient Descent step using tf.GradientTape."""
        with tf.GradientTape() as tape:
            y_pred = self.forward(X)
            loss = self.compute_loss(y, y_pred)
            
        # Compute exact partial gradients
        dW, db = tape.gradient(loss, [self.W, self.b])
        
        # In-place parameter updates: theta = theta - lr * grad
        self.W.assign_sub(self.lr * dW)
        self.b.assign_sub(self.lr * db)
        
        return loss

    def fit(self, X, y, epochs=500, print_every=100):
        self.loss_history = []
        for epoch in range(1, epochs + 1):
            loss = self.train_step(X, y)
            self.loss_history.append(loss.numpy())
            
            if epoch % print_every == 0 or epoch == 1:
                preds = (self.forward(X) >= 0.5)
                acc = tf.reduce_mean(tf.cast(preds == tf.cast(y, tf.bool), tf.float32)).numpy() * 100
                print(f"Epoch {epoch:4d}/{epochs} | Loss: {loss.numpy():.5f} | Accuracy: {acc:.2f}%")
        return self

    def predict(self, X):
        probs = self.forward(X)
        return tf.cast(probs >= 0.5, tf.int32).numpy()

    def predict_proba(self, X):
        return self.forward(X).numpy()


#### **Step 5: Training the Low-Level TensorFlow Model**


In [ ]:
print("=== TRAINING LOW-LEVEL TENSORFLOW LOGISTIC REGRESSION ===")
low_level_model = LowLevelLogisticRegression(n_features=2, learning_rate=0.2, seed=42)
low_level_model.fit(X_train_tf, y_train_tf, epochs=500, print_every=100)

learned_W = low_level_model.W.numpy().flatten()
learned_b = low_level_model.b.numpy()[0]
print(f"
[FINAL PARAMETERS] Learned Weights W: {learned_W}, Bias b: {learned_b:.4f}")


#### **Step 6: High-Level Keras Implementation**
We construct the identical logistic model using TensorFlow's high-level Keras abstraction:
- Architecture: `tf.keras.Sequential([tf.keras.layers.Dense(1, activation='sigmoid')])`
- Optimization: `Adam(learning_rate=0.05)`
- Loss: `'binary_crossentropy'`


In [ ]:
# High-Level Keras Architecture
keras_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(1, activation='sigmoid', name="Logistic_Neuron")
], name="Keras_Logistic_Regression")

keras_model.summary()

# Compilation
keras_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Training
print("
--- Training High-Level Keras Model ---")
keras_history = keras_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=32,
    verbose=0
)

print("[SUCCESS] High-Level Keras Model trained successfully across 100 epochs!")
keras_weights, keras_bias = keras_model.layers[0].get_weights()
print(f"[FINAL PARAMETERS] Keras Weights W: {keras_weights.flatten()}, Bias b: {keras_bias[0]:.4f}")


#### **Step 7: Evaluation, Metric Benchmarking & Comparison**
Evaluate both implementations on the test set ($\mathbf{X}_{	ext{test}}$) across all clinical metrics.


In [ ]:
# Low-Level Model Predictions
low_level_preds = low_level_model.predict(X_test_tf).flatten()
low_level_probs = low_level_model.predict_proba(X_test_tf).flatten()

# High-Level Keras Model Predictions
keras_probs = keras_model.predict(X_test_scaled, verbose=0).flatten()
keras_preds = (keras_probs >= 0.5).astype(int)

# Metric Computation
comparison_data = [
    {
        "Framework": "Low-Level TensorFlow (Custom Loop)",
        "Accuracy": accuracy_score(y_test, low_level_preds),
        "Precision": precision_score(y_test, low_level_preds),
        "Recall": recall_score(y_test, low_level_preds),
        "F1-Score": f1_score(y_test, low_level_preds),
        "ROC-AUC": roc_auc_score(y_test, low_level_probs)
    },
    {
        "Framework": "High-Level TensorFlow (Keras Sequential)",
        "Accuracy": accuracy_score(y_test, keras_preds),
        "Precision": precision_score(y_test, keras_preds),
        "Recall": recall_score(y_test, keras_preds),
        "F1-Score": f1_score(y_test, keras_preds),
        "ROC-AUC": roc_auc_score(y_test, keras_probs)
    }
]

import pandas as pd
comp_df = pd.DataFrame(comparison_data)
print("=== PERFORMANCE COMPARISON TABLE ===")
print(comp_df.to_string(index=False))


#### **Step 8: Confusion Matrix & ROC Curve Comparison**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Low-Level Confusion Matrix
cm_low = confusion_matrix(y_test, low_level_preds)
sns.heatmap(cm_low, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
axes[0].set_title(f"Low-Level TF Confusion Matrix\nAccuracy: {accuracy_score(y_test, low_level_preds)*100:.2f}%", fontweight='bold')
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# 2. Keras Confusion Matrix
cm_keras = confusion_matrix(y_test, keras_preds)
sns.heatmap(cm_keras, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[1])
axes[1].set_title(f"High-Level Keras Confusion Matrix\nAccuracy: {accuracy_score(y_test, keras_preds)*100:.2f}%", fontweight='bold')
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

# 3. ROC Curve Comparison
fpr_low, tpr_low, _ = roc_curve(y_test, low_level_probs)
fpr_k, tpr_k, _ = roc_curve(y_test, keras_probs)

axes[2].plot(fpr_low, tpr_low, label=f"Low-Level TF (AUC = {roc_auc_score(y_test, low_level_probs):.4f})", color='#2b5c8f', lw=2)
axes[2].plot(fpr_k, tpr_k, label=f"Keras Model (AUC = {roc_auc_score(y_test, keras_probs):.4f})", color='#2ca02c', lw=2, linestyle='--')
axes[2].plot([0, 1], [0, 1], color='gray', linestyle=':')
axes[2].set_title("ROC Curves Comparison", fontsize=12, fontweight='bold')
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


#### **Step 9: Visualizing the Learned Linear Decision Boundary**
The learned decision boundary corresponds to:
$$w_1 x_1 + w_2 x_2 + b = 0 \implies x_2 = -rac{w_1}{w_2} x_1 - rac{b}{w_2}$$
We plot this separating hyperplane over the test dataset points.


In [ ]:
plt.figure(figsize=(9, 6.5))

# Scatter actual test points
plt.scatter(
    X_test_scaled[y_test == 0, 0], X_test_scaled[y_test == 0, 1],
    color='#e05d44', label='True Class 0', alpha=0.8, edgecolors='k', s=50
)
plt.scatter(
    X_test_scaled[y_test == 1, 0], X_test_scaled[y_test == 1, 1],
    color='#2b5c8f', label='True Class 1', alpha=0.8, edgecolors='k', s=50
)

# Plot Low-Level TF Decision Boundary Line
x1_vals = np.linspace(X_test_scaled[:, 0].min() - 0.5, X_test_scaled[:, 0].max() + 0.5, 200)
x2_low = -(learned_W[0] * x1_vals + learned_b) / learned_W[1]
plt.plot(x1_vals, x2_low, color='blue', lw=2.5, label='Low-Level TF Decision Boundary')

# Plot Keras Decision Boundary Line
x2_keras = -(keras_weights[0, 0] * x1_vals + keras_bias[0]) / keras_weights[1, 0]
plt.plot(x1_vals, x2_keras, color='green', lw=2, linestyle='--', label='Keras Decision Boundary')

plt.xlim(X_test_scaled[:, 0].min() - 0.5, X_test_scaled[:, 0].max() + 0.5)
plt.ylim(X_test_scaled[:, 1].min() - 0.5, X_test_scaled[:, 1].max() + 0.5)
plt.title("Learned Decision Boundary Hyperplane Comparison", fontsize=13, fontweight='bold')
plt.xlabel("Feature 1 (Standardized)")
plt.ylabel("Feature 2 (Standardized)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
### **5. Comparative Discussion: Low-Level TF vs High-Level Keras**

| Dimension | Low-Level TensorFlow (`tf.GradientTape`) | High-Level Keras (`model.fit`) |
| :--- | :--- | :--- |
| **Control & Customization** | **Absolute control** over gradient updates, loss equations, and custom architectures. | Constrained to standard layers, optimizers, and callback interfaces. |
| **Code Length** | More verbose; requires explicit loop and parameter tracking. | Extremely concise, clean, and declarative. |
| **Debugging** | Can place breakpoints inside gradients and inspect intermediate tensors easily. | Abstraction hides intermediate graph transformations. |
| **Production Readiness** | Requires manual export handling. | Seamless serialization via `model.save()`, TFLite conversion, and serving. |
| **Use Cases** | Novel research, custom loss formulations, physics-informed neural networks (PINNs). | Rapid prototyping, standard computer vision, NLP, and production pipelines. |

---

### **6. Viva-Voce / Review Questions with Answers**

**Q1: Why is Binary Cross-Entropy used instead of Mean Squared Error in Logistic Regression?**
* **Answer**: If Mean Squared Error is used with the Sigmoid activation function, the resulting loss function with respect to weights is **non-convex**, possessing numerous local minima and regions where gradients vanish. Binary Cross-Entropy (derived via Maximum Likelihood Estimation) yields a strictly **convex** optimization landscape, guaranteeing convergence to the global minimum.

**Q2: What is the relationship between Logistic Regression and a single Artificial Neuron?**
* **Answer**: A Logistic Regression model is mathematically identical to a single artificial neuron with a Sigmoid activation function. When multiple such neurons are arranged in parallel and cascaded in sequence, they form a Multi-Layer Perceptron (Deep Neural Network).

**Q3: How does `tf.GradientTape()` compute gradients in TensorFlow?**
* **Answer**: `tf.GradientTape` records all tensor operations executed within its Python context block onto a tape. During the backward pass, TensorFlow uses **Reverse-Mode Automatic Differentiation** to traverse the recorded computational graph in reverse order, applying the chain rule to compute exact partial derivatives without manual mathematical derivation.

**Q4: How is the Decision Boundary mathematically determined in Logistic Regression?**
* **Answer**: The decision boundary is the locus of points where the model outputs an equal probability of $0.5$ for both classes ($\hat{y} = 0.5$). Since $\sigma(z) = 0.5$ when $z = 0$, the decision boundary corresponds to the linear equation $\mathbf{w}^T \mathbf{x} + b = 0$. In two dimensions, this forms the straight line $x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$.

**Q5: What is the purpose of numerical clipping (`tf.clip_by_value`) in the Binary Cross-Entropy loss?**
* **Answer**: The cross-entropy loss computes $\ln(\hat{y})$ and $\ln(1 - \hat{y})$. If the predicted probability $\hat{y}$ reaches exactly $0.0$ or $1.0$, $\ln(0)$ evaluates to $-\infty$, producing `NaN` (Not a Number) gradients that corrupt all model weights. Clipping $\hat{y}$ into $[\epsilon, 1 - \epsilon]$ (where $\epsilon \approx 10^{-15}$) guarantees finite, stable loss and gradients.

---

### **7. Conclusion**
* In this experiment, Logistic Regression for binary classification was successfully implemented using TensorFlow.
* The model was built and trained from scratch using **Low-Level TensorFlow** primitives (`tf.Variable`, `tf.GradientTape`, custom cross-entropy, manual gradient descent).
* The identical architecture was built and trained using **High-Level TensorFlow Keras** (`Dense`, `Sequential`, `Adam`).
* Both implementations converged to virtually identical weights, achieving test accuracies exceeding 95% and ROC-AUC scores exceeding 0.98.
* Confusion matrices, ROC curves, and the linear separating decision boundary were visualized and analyzed.
